# Task Arithmetic Evaluation for ViTs MergeBackdoor

Initialize args for merging.

In [1]:
import argparse
import os
os.environ["CUDA_VISIBLE_DEVICES"] = '3,4'
parser = argparse.ArgumentParser(description='Evaluate the effectiveness of MergeBackdoor with task arithmetic.')
parser.add_argument('--dataset1', type=str, default='CIFAR10', help='first upstream model trained for merge backdoor')
parser.add_argument('--dataset2', type=str, default='MNIST', help='second upstream model trained for merge backdoor')
parser.add_argument('--nb_classes1', type=int, default=10, help='class number of the first task')
parser.add_argument('--nb_classes2', type=int, default=10, help='class number of the second task')
parser.add_argument('--dataset', default='CIFAR10', help='Which dataset to load')
parser.add_argument('--dataset_type', default='CV', help='The dataset belongs to the domain of (CV or NLP)')
parser.add_argument('--epochs', default=5, help='Number of epochs to fine-tune models, default: 5')
parser.add_argument('--batch_size', type=int, default=120, help='Batch size to split dataset, default: 120')
parser.add_argument('--num_workers', type=int, default=0, help='Batch size to split dataset')
parser.add_argument('--data_path', default='./data/', help='Place to load dataset')
parser.add_argument('--poisoning_rate', type=float, default=0.1, help='poisoning rate')
parser.add_argument('--trigger_label', type=int, default=1, help='The NO. of trigger label')
parser.add_argument('--trigger_path', default="./triggers/trigger_white.png", help='Trigger Path')
parser.add_argument('--trigger_size', type=int, default=5, help='Trigger Size')
parser.add_argument("-f", "--fff", help="a dummy argument to fool ipython", default="1")

args = parser.parse_args()

## Model Loading
Load model from fine-tuning saved directory.

In [2]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

dataset1 = "CIFAR10"
dataset2 = "MNIST"
nb_classes1 = 10 # MNIST has 10 output classes
nb_classes2 = 10 # CIFAR-10 has 10 output classes

Initialize merged models.

In [3]:

from models.finetune_vit import ImageEncoder, ImageClassifier
# fine-tuned checkpoints
model1_path = f'./checkpoints/ViT-{dataset1}-mbd.pth'
model2_path = f'./checkpoints/ViT-{dataset2}-mbd.pth'          
model3_1_temp_path = './checkpoints/ViT-temp-model3_1.pth'
model3_2_temp_path = './checkpoints/ViT-temp-model3_2.pth'

# upstream models
print('Building ViTs')
# for merged models
image_encoder11 = ImageEncoder(keep_lang=False)
model3_1 = ImageClassifier(image_encoder11,nb_classes1)
model3_1 = torch.nn.DataParallel(model3_1).to(device)

image_encoder22 = ImageEncoder(keep_lang=False)
model3_2 = ImageClassifier(image_encoder22,nb_classes2)
model3_2 = torch.nn.DataParallel(model3_2).to(device)

model3_1_check = model3_1.state_dict()
model3_2_check = model3_2.state_dict()

Building ViTs


/data1/wlj/wjj/envs/merge/lib/python3.8/site-packages/open_clip/factory.py:128: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(checkpoint_path, map_lo

Initialize state dict of upstream models.

In [4]:
image_encoder_1 = ImageEncoder(keep_lang=False)
model1 = ImageClassifier(image_encoder_1,nb_classes1)
model1 = torch.nn.DataParallel(model1).to(device)

image_encoder_2 = ImageEncoder(keep_lang=False)
model2 = ImageClassifier(image_encoder_2,nb_classes2)
model2 = torch.nn.DataParallel(model2).to(device)

model1.load_state_dict(torch.load(model1_path))
model2.load_state_dict(torch.load(model2_path))

model1_check = model1.state_dict()
model2_check = model2.state_dict() 
model3_1_check['module.fc1.0.weight'] =  model1_check['module.fc1.0.weight']
model3_1_check['module.fc1.0.bias'] = model1_check['module.fc1.0.bias']
model3_2_check['module.fc1.0.weight'] =  model2_check['module.fc1.0.weight']
model3_2_check['module.fc1.0.bias'] = model2_check['module.fc1.0.bias']
torch.save(model3_1_check, model3_1_temp_path)
torch.save(model3_2_check, model3_2_temp_path)
model3_1_check = torch.load(model3_1_temp_path)
model3_2_check = torch.load(model3_2_temp_path)

/tmp/ipykernel_1489939/771655349.py:9: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model1.load_state_dict(torch.load(model1_path))
/tmp/ipykernel_1489939/771655349.py:10: 

## Excluding Parameters for Merging
For classification tasks, we don't merge those parameters of linear heads.

In [5]:
pretrained_param_dict = {param_name: param_value for param_name, param_value in model3_1.named_parameters()}
        
exclude_param_names_regex = []
for key in pretrained_param_dict:
    if pretrained_param_dict[key].dtype in [torch.int64, torch.uint8]:
        exclude_param_names_regex.append(key)

if not exclude_param_names_regex.count('module.fc1.0.weight'):
    exclude_param_names_regex.append('module.fc1.0.weight')

if not exclude_param_names_regex.count('module.fc1.0.bias'):
    exclude_param_names_regex.append('module.fc1.0.bias')

## Load Datasets

In [6]:
from dataset import build_testset
from torch.utils.data import DataLoader

args.dataset = dataset1
print("\n# load dataset1: %s " % args.dataset)

dataset1_val_clean, dataset1_val_poisoned = build_testset(is_train=False, args=args, transform=image_encoder11.train_preprocess)

data1_loader_val_clean    = DataLoader(dataset1_val_clean,     batch_size=args.batch_size, shuffle=True, num_workers=args.num_workers)
data1_loader_val_poisoned = DataLoader(dataset1_val_poisoned,  batch_size=args.batch_size, shuffle=True, num_workers=args.num_workers)

args.dataset = dataset2
print("\n# load dataset2: %s " % args.dataset)

dataset2_val_clean, dataset2_val_poisoned = build_testset(is_train=False, args=args, transform=image_encoder22.train_preprocess)

data2_loader_val_clean    = DataLoader(dataset2_val_clean,     batch_size=args.batch_size, shuffle=True, num_workers=args.num_workers)
data2_loader_val_poisoned = DataLoader(dataset2_val_poisoned,  batch_size=args.batch_size, shuffle=True, num_workers=args.num_workers)


# load dataset1: CIFAR10 
Transform =  Compose(
    RandomResizedCrop(size=(224, 224), scale=(0.9, 1.0), ratio=(0.75, 1.3333), interpolation=bicubic, antialias=True)
    <function _convert_to_rgb at 0x7fef14c6a550>
    ToTensor()
    Normalize(mean=(0.48145466, 0.4578275, 0.40821073), std=(0.26862954, 0.26130258, 0.27577711))
)
Files already downloaded and verified
Files already downloaded and verified
Files already downloaded and verified
False
Poison 0 over 2000 samples ( poisoning rate 0)
Files already downloaded and verified
Files already downloaded and verified
Files already downloaded and verified
True
Poison 2000 over 2000 samples ( poisoning rate 1.0)
Dataset CIFAR10Poison
    Number of datapoints: 2000
    Root location: ./data/
    Split: Test
    StandardTransform
Transform: Compose(
               RandomResizedCrop(size=(224, 224), scale=(0.9, 1.0), ratio=(0.75, 1.3333), interpolation=bicubic, antialias=True)
               <function _convert_to_rgb at 0x7fef14c6a550>
    

## Merging and Testing

In [ ]:
from model_merging_methods.merging_methods import MergingMethod
from utility import evaluate_badnets

best_scale= -1
best1_TA = 0
best2_TA = 0
best1_ASR = 0
best2_ASR = 0
for scale_number in range(0,16):
    scale = scale_number/10.0
    model3_1.load_state_dict(torch.load(model3_1_temp_path))
    model3_2.load_state_dict(torch.load(model3_2_temp_path))
    model1.load_state_dict(torch.load(model1_path))
    model2.load_state_dict(torch.load(model2_path)) 
    mm = MergingMethod(merging_method_name = 'task_arithmetic')
    model3_1 = mm.get_merged_model(merged_model = model3_1, models_to_merge = [model1, model2], exclude_param_names_regex =  exclude_param_names_regex, scaling_coefficient = scale, models_use_deepcopy = True)     
    model3_2 = mm.get_merged_model(merged_model = model3_2, models_to_merge = [model1, model2], exclude_param_names_regex =  exclude_param_names_regex, scaling_coefficient = scale, models_use_deepcopy = True)         

    torch.cuda.empty_cache()
    test_stats13 = evaluate_badnets(data1_loader_val_clean, data1_loader_val_poisoned, model3_1, device)
    test_stats23 = evaluate_badnets(data2_loader_val_clean, data2_loader_val_poisoned, model3_2, device)

    print(scale)
    print(f"# merged model1 {dataset1}_Test Acc: {test_stats13['clean_acc']:.4f}, {dataset1}_ASR: {test_stats13['asr']:.4f}\n")
    print(f"# merged model2 {dataset2}_Test Acc: {test_stats23['clean_acc']:.4f}, {dataset2}_ASR: {test_stats23['asr']:.4f}\n")

    if best_scale == -1 or test_stats13['clean_acc'] + test_stats23['clean_acc'] > best1_TA + best2_TA:
        best_scale = scale
        best1_TA = test_stats13['clean_acc']
        best1_ASR = test_stats13['asr']
        best2_TA = test_stats23['clean_acc']
        best2_ASR = test_stats23['asr']

    print(f"# best_scale: {best_scale}")
    print(f"# best merged model1 TA: {best1_TA:.4f}")
    print(f"# best merged model1 ASR: {best1_ASR:.4f}")
    print(f"# best merged model2 TA:: {best2_TA:.4f}")
    print(f"# best merged model2 ASR:: {best2_ASR:.4f}")

/tmp/ipykernel_1489939/2298562392.py:11: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model3_1.load_state_dict(torch.load(model3_1_temp_path))
/tmp/ipykernel_1489939/229856

              precision    recall  f1-score   support

    airplane       0.01      0.02      0.01       210
  automobile       0.06      0.02      0.03       210
        bird       0.17      0.49      0.26       193
         cat       0.17      0.06      0.09       188
        deer       0.03      0.02      0.03       203
         dog       0.05      0.06      0.05       212
        frog       0.00      0.00      0.00       208
       horse       0.00      0.00      0.00       187
        ship       0.08      0.03      0.05       190
       truck       0.00      0.00      0.00       199

    accuracy                           0.07      2000
   macro avg       0.06      0.07      0.05      2000
weighted avg       0.06      0.07      0.05      2000



  0%|          | 0/17 [00:00<?, ?it/s]/data1/wlj/wjj/envs/merge/lib/python3.8/site-packages/torch/nn/parallel/parallel_apply.py:79: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.device(device), torch.cuda.stream(stream), autocast(enabled=autocast_enabled):
  0%|          | 0/17 [00:00<?, ?it/s]/data1/wlj/wjj/envs/merge/lib/python3.8/site-packages/torch/nn/parallel/parallel_apply.py:79: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.device(device), torch.cuda.stream(stream), autocast(enabled=autocast_enabled):
100%|██████████| 17/17 [00:09<00:00,  1.76it/s]
/data1/wlj/wjj/envs/merge/lib/python3.8/site-packages/sklearn/metrics/_classification.py:1469: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to 

              precision    recall  f1-score   support

    0 - zero       0.00      0.00      0.00       207
     1 - one       0.00      0.00      0.00       217
     2 - two       0.00      0.00      0.00       197
   3 - three       0.60      0.01      0.03       201
    4 - four       0.00      0.00      0.00       197
    5 - five       0.00      0.00      0.00       182
     6 - six       0.00      0.00      0.00       203
   7 - seven       0.06      0.24      0.09       228
   8 - eight       0.08      0.47      0.14       187
    9 - nine       0.00      0.00      0.00       181

    accuracy                           0.07      2000
   macro avg       0.07      0.07      0.03      2000
weighted avg       0.07      0.07      0.03      2000



  0%|          | 0/17 [00:00<?, ?it/s]/data1/wlj/wjj/envs/merge/lib/python3.8/site-packages/torch/nn/parallel/parallel_apply.py:79: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.device(device), torch.cuda.stream(stream), autocast(enabled=autocast_enabled):
100%|██████████| 17/17 [00:10<00:00,  1.69it/s]
/tmp/ipykernel_1489939/2298562392.py:11: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via

...

  0%|          | 0/17 [00:00<?, ?it/s]/data1/wlj/wjj/envs/merge/lib/python3.8/site-packages/torch/nn/parallel/parallel_apply.py:79: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.device(device), torch.cuda.stream(stream), autocast(enabled=autocast_enabled):
  0%|          | 0/17 [00:00<?, ?it/s]/data1/wlj/wjj/envs/merge/lib/python3.8/site-packages/torch/nn/parallel/parallel_apply.py:79: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.device(device), torch.cuda.stream(stream), autocast(enabled=autocast_enabled):
100%|██████████| 17/17 [00:09<00:00,  1.71it/s]


              precision    recall  f1-score   support

    0 - zero       0.99      1.00      0.99       207
     1 - one       0.93      1.00      0.96       217
     2 - two       0.97      0.99      0.98       197
   3 - three       1.00      1.00      1.00       201
    4 - four       1.00      0.99      0.99       197
    5 - five       0.98      1.00      0.99       182
     6 - six       0.99      0.98      0.99       203
   7 - seven       1.00      0.91      0.95       228
   8 - eight       0.99      0.98      0.99       187
    9 - nine       0.98      0.99      0.99       181

    accuracy                           0.98      2000
   macro avg       0.98      0.98      0.98      2000
weighted avg       0.98      0.98      0.98      2000



  0%|          | 0/17 [00:00<?, ?it/s]/data1/wlj/wjj/envs/merge/lib/python3.8/site-packages/torch/nn/parallel/parallel_apply.py:79: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.device(device), torch.cuda.stream(stream), autocast(enabled=autocast_enabled):
100%|██████████| 17/17 [00:10<00:00,  1.67it/s]

1.5
# merged model1 CIFAR10_Test Acc: 0.7550, CIFAR10_ASR: 0.1820

# merged model2 MNIST_Test Acc: 0.9825, MNIST_ASR: 0.2870

# best_scale: 0.4
# best merged model1 TA: 0.9900
# best merged model1 ASR: 0.9725
# best merged model2 TA:: 0.9925
# best merged model2 ASR:: 0.9960


: 